# 12장 학습 파이프라인 한 바퀴 — MNIST와 MLP

**1단계 · 따라 하기** — 『법률가를 위한 인공지능 코딩 자습서』 실습 노트북

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch12_1_follow.ipynb)

이 노트북에는 책의 코드가 순서대로 들어 있습니다. **위에서부터 셀을 하나씩 실행**하고(셀을 누른 뒤 `Shift+Enter`), 출력이 책과 같은지 대조하세요.

- 셀 아래에 미리 보이는 출력은 저자의 컴퓨터(CPU)에서 실행한 결과입니다. 걸린 시간은 환경마다 다릅니다.
- 책은 코드를 **붙여 넣지 말고 직접 치라**고 권합니다. 실행해 본 다음, 셀 아래에 빈 코드 셀을 만들어(`+ 코드`) 같은 코드를 눈으로 읽으며 쳐 보세요.
- 설명은 책에 있습니다. 이 노트북은 실행을 위한 것입니다.

| 단계 | 할 일 | 노트북 |
|---|---|---|
| 1. 따라 하기 | `train_mnist.py` 를 12.0의 주석 여섯 줄을 먼저 치고, 그 아래를 채우는 순서로 입력. 실행해 정확도 0.94 근처 확인 | [ch12_1_follow.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch12_1_follow.ipynb) **(지금 이 노트북)** |
| 2. 주석 달기 | 반복문 안 다섯 줄에 ①~⑤를, `evaluate` 의 세 장치(`no_grad`·`eval`·`argmax`)에 이유를 주석으로 | [ch12_2_comment.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch12_2_comment.ipynb) |
| 3. 다시 만들기 | 주석 여섯 줄만 남긴 빈 파일에서 전체를 복원. 그다음 `experiments.py` 의 `run` 함수처럼 감싸서 학습률 세 개를 비교 | [ch12_3_build.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch12_3_build.ipynb) |

In [ ]:
# 준비 셀 — 맨 먼저 한 번 실행하세요. 실습 코드와 데이터를 받아 옵니다.
import os, sys
if "google.colab" in sys.modules:                       # Colab 에서 열었을 때
    ROOT = "/content/AI-Coding-for-Lawyers"
    if not os.path.isdir(ROOT):
        !git clone -q --depth 1 https://github.com/BowonKwon/AI-Coding-for-Lawyers.git {ROOT}
else:                                                   # 내 컴퓨터에서 열었을 때 (저장소 폴더 안)
    ROOT = os.getcwd()
    while not os.path.isdir(os.path.join(ROOT, "notebooks")) and os.path.dirname(ROOT) != ROOT:
        ROOT = os.path.dirname(ROOT)
os.environ.setdefault("JUCOS_DATA", os.path.join(ROOT, "data"))   # 데이터를 두는 폴더
os.chdir(os.path.join(ROOT, "code", "12"))                      # 이 장의 코드 폴더로 이동
print("준비 끝. 작업 폴더: code/12")

준비 끝. 작업 폴더: code/12


## 12.0 논리 흐름을 주석으로 먼저

코드를 치기 전에 뼈대를 주석 여섯 줄로 적습니다. 아래 `train_mnist.py` 는 이 여섯 줄을 채운 것입니다.

```python
# 0. 설정 (배치 크기, 학습률, 에포크 수, 씨앗)
# 1. 데이터: MNIST 를 받아 (이미지, 정답) 을 배치로 공급하는 로더 둘 (학습용·시험용)
# 2. 모델: 11장의 MLP (784 → 128 → 10)
# 3. 손실: 분류용 교차 엔트로피
# 4. 옵티마이저: SGD 에 모델의 파라미터와 학습률을 준다
# 5. 반복: 에포크마다 { 배치마다 { ① zero_grad ② 예측 ③ 손실 ④ backward ⑤ step } 시험 정확도 }
```

## `train_mnist.py` — 12장 학습 파이프라인 한 바퀴: MNIST 손글씨 숫자를 MLP로 분류한다.

처음 실행하면 MNIST(약 64MB)를 내려받습니다. Colab 에서 GPU를 켰다면 `device = torch.device("cuda")` 로 바꿔 볼 수 있습니다(이 크기의 모델은 빨라지지 않습니다. 9.5절).

In [ ]:
# 12장 학습 파이프라인 한 바퀴: MNIST 손글씨 숫자를 MLP로 분류한다.
# 준비물 넷(데이터·모델·손실·옵티마이저) + 반복(걸음·에포크) + 평가.
# 실행: python train_mnist.py            (데이터는 DATA_ROOT 에 자동 다운로드, 약 64MB)

import os
import time
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [ ]:
# ---- 0. 설정: 이 장에서는 딕셔너리 하나에 모아 둔다 (13장에서 argparse 로 바꾼다)
config = {
    "data_root": os.environ.get("JUCOS_DATA", "./data"),   # 데이터 폴더. 환경변수 JUCOS_DATA 가 있으면 그것, 없으면 ./data
    "batch_size": 64,
    "hidden": 128,
    "lr": 0.1,
    "epochs": 2,
    "seed": 0,
    "log_every": 200,           # 몇 걸음마다 손실을 찍을지
}
torch.manual_seed(config["seed"])
device = torch.device("cpu")    # Colab에서 GPU를 켰다면 torch.device("cuda")

In [ ]:
# ---- 1. 데이터: (이미지, 정답) 쌍을 배치로 공급한다
transform = transforms.ToTensor()                              # PIL 이미지 → [1, 28, 28] float 텐서 (0~1)
train_set = datasets.MNIST(config["data_root"], train=True, download=True, transform=transform)
test_set = datasets.MNIST(config["data_root"], train=False, download=True, transform=transform)
train_loader = DataLoader(train_set, batch_size=config["batch_size"], shuffle=True)   # 학습은 섞는다
test_loader = DataLoader(test_set, batch_size=1000)                                  # 평가는 순서 무관
print(f"학습 {len(train_set)}장 / 시험 {len(test_set)}장, 배치 {len(train_loader)}개")

학습 60000장 / 시험 10000장, 배치 938개


In [ ]:
# ---- 2. 모델: 11장의 MLP 그대로
class MLP(nn.Module):
    def __init__(self, in_dim, hidden, out_dim):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, out_dim)

    def forward(self, x):
        x = x.flatten(1)                 # [B, 1, 28, 28] → [B, 784]  (모델 안에서 편다)
        x = torch.relu(self.fc1(x))
        return self.fc2(x)               # 점수 10개 (logits)


model = MLP(784, config["hidden"], 10).to(device)

In [ ]:
# ---- 3. 손실: 분류의 표준, 교차 엔트로피 (점수 → 확률 → 정답 확률의 -log)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
# ---- 4. 옵티마이저: 확률적 경사 하강법 (SGD). 파라미터 목록과 학습률을 준다
optimizer = torch.optim.SGD(model.parameters(), lr=config["lr"])


def evaluate(model, loader):
    """시험 데이터 전체에 대한 정확도. 학습하지 않으므로 기울기를 끈다."""
    model.eval()                                   # 평가 모드 (12장에서는 차이 없음, 습관)
    correct = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            scores = model(images)                 # [B, 10]
            pred = scores.argmax(dim=1)            # 가장 큰 점수의 자리 = 예측 숫자
            correct += (pred == labels).sum().item()
    model.train()                                  # 학습 모드로 복귀
    return correct / len(loader.dataset)

In [ ]:
# ---- 5. 반복: 걸음(배치) × 에포크
print(f"학습 전 정확도: {evaluate(model, test_loader):.3f}")
start = time.time()
for epoch in range(1, config["epochs"] + 1):
    running = 0.0
    for step, (images, labels) in enumerate(train_loader, start=1):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()                      # ① 기울기 초기화
        scores = model(images)                     # ② 예측
        loss = loss_fn(scores, labels)             # ③ 손실
        loss.backward()                            # ④ 기울기 계산
        optimizer.step()                           # ⑤ 가중치 갱신
        running += loss.item()
        if step % config["log_every"] == 0:
            print(f"  에포크 {epoch} 걸음 {step:4d}/{len(train_loader)}  평균 손실 {running / config['log_every']:.4f}")
            running = 0.0
    acc = evaluate(model, test_loader)
    print(f"에포크 {epoch} 끝: 시험 정확도 {acc:.3f}  (경과 {time.time() - start:.0f}초)")

학습 전 정확도: 0.104


  에포크 1 걸음  200/938  평균 손실 0.8660


  에포크 1 걸음  400/938  평균 손실 0.3808


  에포크 1 걸음  600/938  평균 손실 0.3290


  에포크 1 걸음  800/938  평균 손실 0.2889


에포크 1 끝: 시험 정확도 0.928  (경과 6초)


  에포크 2 걸음  200/938  평균 손실 0.2454


  에포크 2 걸음  400/938  평균 손실 0.2416


  에포크 2 걸음  600/938  평균 손실 0.2265


  에포크 2 걸음  800/938  평균 손실 0.2048


에포크 2 끝: 시험 정확도 0.946  (경과 11초)


## 12.4 준비물을 하나씩 망가뜨리기

아래 실험 가운데 `zero_grad 빠뜨림` 처럼 **발산하는 실험의 손실과 정확도는 실행 환경에 따라 달라집니다**(이 노트북에 저장된 값은 3.0637 · 0.098 이고, Colab 에서는 2.6686 · 0.124 가 나온 적이 있습니다). 학습이 망가지는데 오류는 나지 않는다는 점은 같습니다. 나머지 여섯 줄의 손실과 정확도는 Colab 에서도 책과 같게 나왔습니다(걸린 시간은 다릅니다).

## `experiments.py` — 12장 실험: 준비물 하나씩 바꿔 보기. train_mnist.py 를 함수로 감싼 판.

이 파일은 준비 셀에서 이미 받아 두었습니다. 내용은 아래와 같습니다. (왼쪽 폴더 모양 아이콘에서 파일을 두 번 누르면 열어서 고칠 수도 있습니다.)

```python
# 12장 실험: 준비물 하나씩 바꿔 보기. train_mnist.py 를 함수로 감싼 판.
# 실행: python experiments.py
import os
import time
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

DATA_ROOT = os.environ.get("JUCOS_DATA", "./data")
torch.manual_seed(0)
transform = transforms.ToTensor()
train_set = datasets.MNIST(DATA_ROOT, train=True, download=True, transform=transform)
test_set = datasets.MNIST(DATA_ROOT, train=False, download=True, transform=transform)
test_loader = DataLoader(test_set, batch_size=1000)


class MLP(nn.Module):
    def __init__(self, in_dim=784, hidden=128, out_dim=10):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, out_dim)

    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x.flatten(1))))


def accuracy(model):
    model.eval()
    correct = 0
    with torch.no_grad():
        for images, labels in test_loader:
            correct += (model(images).argmax(1) == labels).sum().item()
    model.train()
    return correct / len(test_set)


def run(name, lr=0.1, opt="sgd", zero_grad=True, batch_size=64, max_steps=None, seed=0):
    """준비물 하나를 바꿔 1 에포크 학습하고 정확도를 돌려준다."""
    torch.manual_seed(seed)
    loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    model = MLP()
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr) if opt == "sgd" else torch.optim.Adam(model.parameters(), lr=lr)
    t0 = time.time()
    last_losses = []
    for step, (images, labels) in enumerate(loader, start=1):
        if zero_grad:
            optimizer.zero_grad()
        loss = loss_fn(model(images), labels)
        loss.backward()
        optimizer.step()
        last_losses.append(loss.item())
        if max_steps and step >= max_steps:
            break
    tail = sum(last_losses[-50:]) / min(50, len(last_losses))
    acc = accuracy(model)
    print(f"{name:<28} 마지막 50걸음 평균 손실 {tail:8.4f}  정확도 {acc:.3f}  ({time.time() - t0:.0f}초)")
    return acc


if __name__ == "__main__":
    run("기준 (SGD, lr=0.1)")
    run("학습률 너무 작음 (lr=0.001)", lr=0.001)
    run("학습률 너무 큼 (lr=10)", lr=10)
    run("zero_grad 빠뜨림", zero_grad=False)
    run("옵티마이저 Adam (lr=0.001)", lr=0.001, opt="adam")
    run("배치 1 (300걸음만)", batch_size=1, max_steps=300)
    run("배치 60000 (한 걸음)", batch_size=60000)
```

In [ ]:
!python experiments.py

기준 (SGD, lr=0.1)             마지막 50걸음 평균 손실   0.2538  정확도 0.925  (6초)
학습률 너무 작음 (lr=0.001)         마지막 50걸음 평균 손실   2.1032  정확도 0.570  (6초)
학습률 너무 큼 (lr=10)             마지막 50걸음 평균 손실   2.3809  정확도 0.096  (6초)
zero_grad 빠뜨림                마지막 50걸음 평균 손실   3.0637  정확도 0.098  (6초)
옵티마이저 Adam (lr=0.001)        마지막 50걸음 평균 손실   0.1876  정확도 0.945  (7초)
배치 1 (300걸음만)                마지막 50걸음 평균 손실   1.9069  정확도 0.561  (1초)
배치 60000 (한 걸음)              마지막 50걸음 평균 손실   2.3031  정확도 0.183  (4초)


## 다음 단계

- 2단계 「주석 달기」: [ch12_2_comment.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch12_2_comment.ipynb)
- 3단계 「다시 만들기」: [ch12_3_build.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch12_3_build.ipynb)
- 책의 「확인 문제」는 이 노트북 끝에 셀을 추가해 풀어 보세요.